# 02 — Data Cleaning & Pipeline

**Project:** Olist Customer Churn Analysis  
**Capstone:** NST DVA Capstone 2 — Section D, Group 9  
**Notebook Role:** ETL Stage 2 — Clean all 7 raw tables, merge them into a single flat master table, engineer the churn label and key analytical columns, reduce to a representative 50,000–60,000 row sample, and export to `data/processed/olist_churn_master.csv`.

This is the main ETL pipeline. Every transformation step has a markdown cell above it explaining the decision. Tools allowed: pandas, numpy, pathlib, sys. No scipy, no seaborn, no matplotlib yet — those come in NB03.

The output of this notebook — `olist_churn_master.csv` — is the single input for all downstream analysis and Tableau dashboarding.


## 1. Imports & Project Path Setup

We import pandas, numpy, pathlib, and sys. We import `basic_clean` from `scripts/etl_pipeline.py` — this is the template-provided helper that strips whitespace from all string columns, drops duplicates, and normalises column names to snake_case. The `PROJECT_ROOT` pattern uses the template's path-resolution logic so the notebook works from both `/notebooks/` and the repo root.


In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

# Resolve project root whether notebook is run from /notebooks/ or from root
PROJECT_ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().resolve().name == "notebooks"
    else Path.cwd().resolve()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.etl_pipeline import basic_clean

RAW_DIR       = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_PATH = PROCESSED_DIR / "olist_churn_master.csv"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root   : {PROJECT_ROOT}")
print(f"Raw dir        : {RAW_DIR}")
print(f"Processed dir  : {PROCESSED_DIR}")
print(f"Output file    : {PROCESSED_PATH}")


## 2. Load All 7 Raw Datasets

We call `pd.read_csv` seven times — one call per file, each into its own named DataFrame. No importing happens anywhere else in this notebook. We immediately call `basic_clean(df)` on each one. `basic_clean` performs three safe, universal operations:

1. **Strip whitespace** from all string columns — prevents invisible mismatches on join keys.
2. **Drop exact duplicates** — no Olist table should have fully duplicate rows.
3. **Normalise column names** to snake_case — the template function handles this via regex, so `product_name_lenght` survives as-is and we rename it explicitly in Step 3.

We print shape after loading each file to confirm nothing was unexpectedly dropped.


In [ ]:
df_orders    = basic_clean(pd.read_csv(RAW_DIR / "olist_orders_dataset.csv"))
df_customers = basic_clean(pd.read_csv(RAW_DIR / "olist_customers_dataset.csv"))
df_items     = basic_clean(pd.read_csv(RAW_DIR / "olist_order_items_dataset.csv"))
df_payments  = basic_clean(pd.read_csv(RAW_DIR / "olist_order_payments_dataset.csv"))
df_reviews   = basic_clean(pd.read_csv(RAW_DIR / "olist_order_reviews_dataset.csv"))
df_products  = basic_clean(pd.read_csv(RAW_DIR / "olist_products_dataset.csv"))
df_category  = basic_clean(pd.read_csv(RAW_DIR / "product_category_name_translation.csv"))

for name, df in [
    ("orders",    df_orders),
    ("customers", df_customers),
    ("items",     df_items),
    ("payments",  df_payments),
    ("reviews",   df_reviews),
    ("products",  df_products),
    ("category",  df_category),
]:
    print(f"  {name:12s}: {df.shape}")


## 3. Apply basic_clean to Each Dataframe

`basic_clean` from the template normalises column names but does not fix known typos in the Olist products table. We rename them explicitly here. The two typos — `product_name_lenght` and `product_description_lenght` — were documented in NB01 Step 9. Fixing them now means every downstream reference uses the corrected spelling. This is a one-line rename and we log it with a markdown cell so the examiners can see the correction was deliberate.


In [ ]:
df_products = df_products.rename(columns={
    "product_name_lenght"        : "product_name_length",
    "product_description_lenght" : "product_description_length",
})

print("Renamed typo columns in df_products:")
print([c for c in df_products.columns if "length" in c])


## 4. Fix Datetime Columns

All timestamp columns in `df_orders`, `df_items`, and `df_reviews` were loaded as `object` (string) dtype — confirmed in NB01 Step 4. We cast them to `datetime64` using `pd.to_datetime(..., errors='coerce')`.

Using `errors='coerce'` is important — it converts unparseable values to `NaT` rather than raising an exception, making the null audit in the next step meaningful. We document each column being cast.

Five columns in `df_orders`, one in `df_items`, two in `df_reviews`.


In [ ]:
# df_orders — 5 timestamp columns
order_ts_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for col in order_ts_cols:
    df_orders[col] = pd.to_datetime(df_orders[col], errors="coerce")

# df_items — 1 timestamp column
df_items["shipping_limit_date"] = pd.to_datetime(df_items["shipping_limit_date"], errors="coerce")

# df_reviews — 2 timestamp columns
df_reviews["review_creation_date"]    = pd.to_datetime(df_reviews["review_creation_date"],    errors="coerce")
df_reviews["review_answer_timestamp"] = pd.to_datetime(df_reviews["review_answer_timestamp"], errors="coerce")

print("Datetime casting complete.")
print("\ndf_orders dtypes after cast:")
print(df_orders[order_ts_cols].dtypes)


## 5. Filter Orders to 'delivered' Only

This is a critical documented decision. We filter `df_orders` to only rows where `order_status == 'delivered'`.

**Why:** Our churn definition is built around completed order history. Non-delivered orders (cancelled, invoiced, processing, created, approved, unavailable, shipped-but-not-delivered) do not represent completed customer transactions and cannot be used to define churn behaviour reliably. We write a markdown cell justifying how many rows we are dropping and confirm the new shape.

From NB01 Step 6: the delivered subset has 96,478 rows; we expect to drop ~2,963 rows.


In [ ]:
original_order_count = len(df_orders)
df_orders = df_orders[df_orders["order_status"] == "delivered"].copy()
dropped = original_order_count - len(df_orders)

print(f"Before filter : {original_order_count:,} rows")
print(f"After filter  : {len(df_orders):,} rows  (order_status == 'delivered' only)")
print(f"Dropped       : {dropped:,} rows  (non-delivered orders — cancelled, processing, etc.)")
print(f"\nValue counts after filter:\n{df_orders['order_status'].value_counts()}")


## 6. Handle Nulls in df_orders

After filtering to delivered only, we re-check nulls in the order timestamp columns. From NB01:

- `order_approved_at` — ~160 nulls. These are orders where approval was not logged. We fill them with `order_purchase_timestamp` since approval typically follows purchase closely; these are data recording gaps, not genuine business events.
- `order_delivered_carrier_date` — ~1,783 nulls. After filtering to delivered we check how many remain. We drop any rows still null here because a delivered order must have been picked up by a carrier.
- `order_delivered_customer_date` — ~2,965 nulls. Same reasoning — a delivered order must have a delivery date. Null here means the delivery confirmation was not recorded. We drop those rows.

We log both decisions explicitly.


In [ ]:
# Fill order_approved_at nulls with order_purchase_timestamp
approved_nulls_before = df_orders["order_approved_at"].isna().sum()
df_orders["order_approved_at"] = df_orders["order_approved_at"].fillna(
    df_orders["order_purchase_timestamp"]
)
approved_nulls_after = df_orders["order_approved_at"].isna().sum()
print(f"order_approved_at nulls: {approved_nulls_before} → {approved_nulls_after}  (filled with purchase timestamp)")

# Drop rows where order_delivered_carrier_date is null
carrier_nulls = df_orders["order_delivered_carrier_date"].isna().sum()
df_orders = df_orders.dropna(subset=["order_delivered_carrier_date"])
print(f"Dropped {carrier_nulls} rows with null order_delivered_carrier_date")

# Drop rows where order_delivered_customer_date is null
customer_nulls = df_orders["order_delivered_customer_date"].isna().sum()
df_orders = df_orders.dropna(subset=["order_delivered_customer_date"])
print(f"Dropped {customer_nulls} rows with null order_delivered_customer_date")

print(f"\ndf_orders shape after null handling: {df_orders.shape}")
print(f"Remaining nulls:\n{df_orders.isnull().sum()[df_orders.isnull().sum() > 0]}")


## 7. Handle Nulls in df_products

Two issues in `df_products` (from NB01):

- `product_category_name` — 610 nulls. We fill with the string `'unknown'`. These are physical dimension products whose category was not entered in the Olist catalogue. Dropping them would remove valid product entries from the join. `'unknown'` is the correct business placeholder.
- Dimension columns (`product_weight_g`, `product_length_cm`, `product_height_cm`, `product_width_cm`) — 2 nulls. These 2 rows are the same 2 rows missing category names. We drop them. Rows with no dimension data cannot be used for any product-level feature engineering downstream.

We log what `product_category_name` null count looks like after the fill.


In [ ]:
# Fill product_category_name nulls with 'unknown'
cat_nulls = df_products["product_category_name"].isna().sum()
df_products["product_category_name"] = df_products["product_category_name"].fillna("unknown")
print(f"product_category_name nulls filled: {cat_nulls} → {df_products['product_category_name'].isna().sum()}")

# Drop 2 rows with null physical dimension columns
dim_cols = ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]
rows_before = len(df_products)
df_products = df_products.dropna(subset=dim_cols)
print(f"Dropped {rows_before - len(df_products)} rows with null dimension columns")
print(f"df_products shape: {df_products.shape}")


## 8. Handle Nulls in df_reviews

From NB01, `review_comment_title` has 87,656 nulls and `review_comment_message` has 58,247 nulls. These are **expected** null fields — Olist's review system makes comment text optional. Customers are not required to write anything; a numeric score is the only mandatory field.

Dropping these rows would remove the majority of reviews from our dataset, which would break the review score join entirely. The correct action is to fill both columns with the string `'no_comment'`. This makes the null explicit without losing rows, and it signals in downstream analysis that no comment was provided — which is itself a behavioural signal. Two rows with empty string `''` in `review_score` should be explicitly checked; `review_score` itself should have no nulls.


In [ ]:
df_reviews["review_comment_title"] = df_reviews["review_comment_title"].fillna("no_comment")
df_reviews["review_comment_message"] = df_reviews["review_comment_message"].fillna("no_comment")

print(f"review_comment_title  remaining nulls: {df_reviews['review_comment_title'].isna().sum()}")
print(f"review_comment_message remaining nulls: {df_reviews['review_comment_message'].isna().sum()}")
print(f"review_score nulls: {df_reviews['review_score'].isna().sum()}")
print(f"df_reviews shape: {df_reviews.shape}")


## 9. Merge Tables — Step-by-Step

We merge all 7 tables into one flat master dataframe called `df_master`. The merge sequence and join type matters:

1. **df_orders LEFT JOIN df_customers** on `customer_id` — every delivered order has a customer
2. **result LEFT JOIN df_items** on `order_id` — some orders may have multiple items, which increases row count; this is expected
3. **result LEFT JOIN df_payments** on `order_id` — same, payments may be multiple per order
4. **result LEFT JOIN df_products** on `product_id` — links item to product details
5. **result LEFT JOIN df_category** on `product_category_name` — translates category to English
6. **result LEFT JOIN df_reviews** on `order_id` — attaches review score per order

After each merge we print the shape and confirm no explosion or massive column growth. We name the final merged dataframe `df_master`.

**Critical note on join type:** All joins are LEFT from `df_orders`. This preserves every delivered order as the spine. We never use an inner join here because that would silently drop orders with no review or payment record — losing rows we should keep.


In [ ]:
# Merge 1: orders + customers
df_master = df_orders.merge(df_customers, on="customer_id", how="left")
print(f"After merge 1 (orders + customers)    : {df_master.shape}")

# Merge 2: + items (on order_id)
df_master = df_master.merge(df_items, on="order_id", how="left")
print(f"After merge 2 (+ items)                : {df_master.shape}")

# Merge 3: + payments (on order_id)
df_master = df_master.merge(df_payments, on="order_id", how="left")
print(f"After merge 3 (+ payments)             : {df_master.shape}")

# Merge 4: + products (on product_id)
df_master = df_master.merge(df_products, on="product_id", how="left")
print(f"After merge 4 (+ products)             : {df_master.shape}")

# Merge 5: + category translation (on product_category_name)
df_master = df_master.merge(df_category, on="product_category_name", how="left")
print(f"After merge 5 (+ category translation) : {df_master.shape}")

# Merge 6: + reviews (on order_id)
df_master = df_master.merge(df_reviews, on="order_id", how="left")
print(f"After merge 6 (+ reviews)              : {df_master.shape}")
